# Model Compression

- 如何选择压缩技术？
  - 权重剪枝：适合需要减少计算量和加速推理的场景。
  - 权重共享：适合内存资源受限但计算能力较强的设备。
  - 知识蒸馏：适用于大模型无法部署的场景，通过训练更小的模型来达到接近原始模型的效果。（感觉不适合现在这个模型，或者说我不会用这种方法）
  - 低秩分解：适合对矩阵计算复杂度高的模型（如卷积层）进行优化。
  - 量化：适合需要减少模型存储需求和提升推理速度的场景，尤其在边缘设备上。

In [23]:
import json

import numpy as np
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel, AdamW
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
class Arguments(object):
    bert_dir = '/Users/bowie/Documents/muti-model/bert-base-chinese'
    data_dir = '/Users/bowie/Documents/muti-data/nlp-意图识别/SMP2019/data.json'

    # 找出intent和slot
    # 以及text的最大长度
    with open(data_dir, 'r', encoding="utf-8") as f:
        data = json.load(f)
    
    intent_lst = []
    slot_lst = []
    slot_ner = ['O']
    text_max_len = 0

    for i in data:
        if i['intent'] not in intent_lst:
            intent_lst.append(i['intent'])
        for k,v in i['slots'].items():
            if k not in slot_lst:
                slot_lst.append(k)
                slot_ner.append(f'B-{k}')
                slot_ner.append(f'I-{k}')
        if len(i['text']) > text_max_len:
            text_max_len = len(i['text'])

    intent_num = len(intent_lst)
    slot_num = len(slot_ner)

    print('intent为: ', intent_num, intent_lst)
    print('slot为: ', len(slot_lst), slot_lst)
    print('slot_ner为: ', slot_num, slot_ner)
    print('text的最大长度为: ', text_max_len)

    # intent的id映射
    id2cls = {}
    cls2id = {}
    for ind, val in enumerate(intent_lst):
        id2cls[ind] = val
        cls2id[val] = ind
    print('id2cls: ', id2cls)
    print('cls2id: ', cls2id)
    
    # ner的id映射
    id2label = {}
    label2id = {}
    for ind, val in enumerate(slot_ner):
        id2label[ind] = val
        label2id[val] = ind

    print('id2label: ', id2label)
    print('label2id: ', label2id)
    # 2.参数设置
    hidden_size = 768
    hidden_dropout = 0.1
    max_len = 32
    # batch_size = 32
    lr=5e-5
    epoch = 50


args = Arguments()

intent为:  23 ['LAUNCH', 'QUERY', 'ROUTE', 'SENDCONTACTS', 'SEND', 'REPLY', 'REPLAY_ALL', 'LOOK_BACK', 'NUMBER_QUERY', 'POSITION', 'PLAY', 'DEFAULT', 'DIAL', 'TRANSLATION', 'OPEN', 'CREATE', 'FORWARD', 'VIEW', 'SEARCH', 'RISERATE_QUERY', 'DOWNLOAD', 'DATE_QUERY', 'CLOSEPRICE_QUERY']
slot为:  60 ['name', 'Dest', 'Src', 'endLoc_city', 'theatre', 'datetime_time', 'receiver', 'dishName', 'ingredient', 'utensil', 'content', 'datetime_date', 'tvchannel', 'category', 'startLoc_city', 'startDate_date', 'startLoc_poi', 'keyword', 'location_province', 'endLoc_poi', 'endLoc_province', 'location_area', 'location_poi', 'endLoc_area', 'type', 'song', 'artist', 'location_city', 'media', 'popularity', 'author', 'dynasty', 'queryField', 'code', 'startLoc_area', 'startDate_time', 'target', 'resolution', 'tag', 'area', 'film', 'relIssue', 'absIssue', 'headNum', 'teleOperator', 'timeDescr', 'scoreDescr', 'subfocus', 'homeName', 'awayName', 'location_country', 'yesterday', 'startLoc_province', 'episode', 'ar

In [3]:
class IntentClassifier(nn.Module):
    def __init__(self, config):
        super(IntentClassifier, self).__init__()
        self.bert = BertModel.from_pretrained(config.bert_dir)

        # Sequential 类似 keras 的容器，可以放多个层 可以放激活函数
        self.intent_classification = nn.Sequential(
            nn.Dropout(config.hidden_dropout),
            nn.Linear(config.hidden_size, config.intent_num),
        )
        self.slot_ner = nn.Sequential(
            nn.Dropout(config.hidden_dropout),
            nn.Linear(config.hidden_size, config.slot_num)
        )

    def forward(self, input_ids, attention_mask, token_type_ids):
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        # 这是 BERT 的池化输出，通常是代表整个序列的嵌入，适用于序列级任务，比如意图分类。池化输出是 BERT 对输入序列的整体表示。
        pooler_output = bert_output[1]
        # 这是 BERT 的最后一层输出，对每个 token 都有一个表示，适用于 token 级任务，比如槽位填充。这个输出保留了每个 token 的详细信息。
        token_output = bert_output[0]

        intent_logits = self.intent_classification(pooler_output)
        slot_logits = self.slot_ner(token_output)
        return intent_logits, slot_logits
    

In [4]:
model = IntentClassifier(config=args)

A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Pl

In [5]:
# https://blog.csdn.net/weixin_40522801/article/details/106563354
state_dict = torch.load('model50.pt')

/var/folders/yv/w2fs9dl54jq77tx2dt62hxhm0000gn/T/ipykernel_19463/4142111508.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('model50.pt')


In [6]:
model.load_state_dict(state_dict)

<All keys matched successfully>

In [7]:
model.intent_classification[1]

Linear(in_features=768, out_features=23, bias=True)

### 1. 权重剪枝（Pruning）
- 原理：去除模型中不重要或冗余的参数（权重），如将权重值非常接近零的神经元连接剪掉，从而减少模型的计算量和存储需求。
- 优点：剪枝后模型的推理速度加快，存储需求减少。
- 缺点：如果剪枝过度，模型精度可能会下降。
- 常用方法：
  - 全局剪枝：全模型范围内根据权重的绝对值进行剪枝。
  - 结构化剪枝：按层或块剪枝，以保持硬件友好性（如卷积层的过滤器剪枝）。


In [8]:
# 查看剪枝前的权重分布
weights_before = model.intent_classification[1].weight.data.clone()
print("剪枝前的权重统计：")
print(f"均值: {weights_before.mean().item()}, 方差: {weights_before.var().item()}")


剪枝前的权重统计：
均值: -0.00033678379259072244, 方差: 0.0006973535055294633


In [23]:
# 非结构化剪枝 对模型中的某一层进行剪枝 还有一个 结构化剪枝 不重要
# prune.l1_unstructured(model.intent_classification[1], name='weight', amount=0.2)  # 剪枝20%的权重


In [9]:
# 全局剪枝
parameters_to_prune = [
    (model.intent_classification[1], 'weight'),
    (model.slot_ner[1], 'weight'),
]

prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.2,  # 剪掉全局范围内20%的权重
)


In [10]:
# 查看剪枝后的权重分布
weights_after = model.intent_classification[1].weight.data.clone()
print("剪枝后的权重统计：")
print(f"均值: {weights_after.mean().item()}, 方差: {weights_after.var().item()}")

# 打印剪枝前后的权重
print("剪枝前的权重：", weights_before)
print("剪枝后的权重：", weights_after)

# 计算和打印剪枝比例
pruned_weights = weights_before[weights_after == 0]
print("剪枝掉的权重：", pruned_weights)
print(f"剪枝掉的权重数量: {len(pruned_weights)}, 比例: {len(pruned_weights) / len(weights_before) * 100:.2f}%")

剪枝后的权重统计：
均值: -0.0003314979840070009, 方差: 0.0006906169001013041
剪枝前的权重： tensor([[ 0.0217, -0.0110,  0.0002,  ...,  0.0237, -0.0275, -0.0174],
        [ 0.0158, -0.0096,  0.0040,  ..., -0.0059, -0.0367, -0.0092],
        [-0.0436,  0.0125,  0.0081,  ..., -0.0159,  0.0277, -0.0490],
        ...,
        [ 0.0318, -0.0379,  0.0299,  ...,  0.0047,  0.0391,  0.0107],
        [-0.0363, -0.0057,  0.0219,  ..., -0.0238,  0.0010, -0.0187],
        [ 0.0249,  0.0214, -0.0289,  ..., -0.0111,  0.0009,  0.0068]])
剪枝后的权重： tensor([[ 0.0217, -0.0110,  0.0000,  ...,  0.0237, -0.0275, -0.0174],
        [ 0.0158, -0.0096,  0.0000,  ..., -0.0000, -0.0367, -0.0000],
        [-0.0436,  0.0125,  0.0000,  ..., -0.0159,  0.0277, -0.0490],
        ...,
        [ 0.0318, -0.0379,  0.0299,  ...,  0.0000,  0.0391,  0.0107],
        [-0.0363, -0.0000,  0.0219,  ..., -0.0238,  0.0000, -0.0187],
        [ 0.0249,  0.0214, -0.0289,  ..., -0.0111,  0.0000,  0.0000]])
剪枝掉的权重： tensor([0.0002, 0.0047, 0.0080,  ..., 0.0025

In [12]:
tokenizer = BertTokenizer.from_pretrained(args.bert_dir)

/Users/bowie/anaconda3/envs/good/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [13]:
def tag_text(data):
    text = data['text']
    slots = data['slots']
    intent = data['intent']
    # tokens = list(text)  # 这里以字符为单位，如果需要以词为单位，可以修改为分词方式
    tokens = tokenizer.tokenize(text)
    # print(len(tokens), tokens)
    tags = [0] * len(tokens)  # 初始化所有的标签为 'O'


    def find_sublist(a, b):
        len_b = len(b)
        for i in range(len(a) - len_b + 1):
            if a[i:i+len_b] == b:  # 判断从索引 i 开始的子列表是否等于 b
                start_idx = i
                end_idx = i + len_b - 1
                return start_idx, end_idx
    
    for slot_name, slot_value in slots.items():
        # 找到 slot 值在文本中的起始位置
        slot_token = tokenizer.tokenize(slot_value)
        start_idx, end_idx = find_sublist(tokens, slot_token)

        # tags[start_idx] = f'B-{slot_name}'  # 起始位置打 'B-'
        tags[start_idx] = args.label2id[f'B-{slot_name}']
        # 去掉第一个字 然后range在加一个数
        for i in range(start_idx+1, end_idx+1):
            # tags[i] = f'I-{slot_name}'  # 后续位置打 'I-'
            tags[i] = args.label2id[f'I-{slot_name}']
    
    # print(len(tags), tags)
    # 对齐标签和 token 序列的长度
    if len(tags) < args.max_len:
        # 101 + tags + 102 + 0
        tags = [0] + tags + [0] * (args.max_len - len(tags) - 1)
    # return tokens, tags
    inputs = tokenizer(
        text=text,              # 要编码的输入文本，即 i['text']，其中 i 是包含文本信息的字典
        max_length=args.max_len,      # 设定编码后的最大长度。token 序列会被截断或填充到这个长度
        padding='max_length',         # 对 token 序列进行填充（padding）以使其达到 max_length 长度。如果不足，就用填充符号（如 0）补足
        truncation='only_first',      # 如果输入的 token 序列超出了 max_length 长度，裁剪（截断）文本中的第一个句子（常用于句子对的输入）
        return_attention_mask=True,   # 返回 attention_mask，用于指示哪些 token 是实际文本，哪些是填充（1 表示文本 token，0 表示填充值）
        return_token_type_ids=True,   # 返回 token_type_ids，用于区分句子对（BERT 使用 token_type_ids 来区分不同的句子）
    )

    '''
    requires_grad=False：这是张量的一个属性，指定该张量是否需要在训练时计算梯度。

    当 requires_grad=False 时，PyTorch 不会为这个张量计算梯度（即它不会在反向传播过程中更新）。这种设置通常用于模型的输入，或者用于不需要梯度的变量（例如在推理阶段）。
    如果设置为 True，这个张量将在反向传播中计算梯度，用于更新参数（通常在训练阶段需要）。
    '''
    return {
        'text': text, # 排查问题用的
        'input_ids': torch.tensor(inputs['input_ids'], requires_grad=False),
        'attention_mask': torch.tensor(inputs['attention_mask'], requires_grad=False),
        'token_type_ids': torch.tensor(inputs['token_type_ids'], requires_grad=False),
        'seq_label_ids': torch.tensor(args.cls2id[intent], requires_grad=False),
        'token_label_ids': torch.tensor(tags, requires_grad=False),
    }


In [14]:
with open('train.json', 'r', encoding="utf-8") as f:
    train_data = json.load(f)

train_data_lst = [tag_text(i) for i in train_data]

In [16]:
# 自定义数据集类
class CAIDataset(Dataset):
    def __init__(self, data):
        # 初始化数据和标签
        self.data = data

    def __len__(self):
        # 返回数据集的大小
        return len(self.data)

    def __getitem__(self, idx):
        # 根据索引返回一个样本及其对应的标签
        sample = {
            'text': self.data[idx]['text'],
            'input_ids': self.data[idx]['input_ids'],
            'attention_mask': self.data[idx]['attention_mask'],
            'token_type_ids': self.data[idx]['token_type_ids'],
            'seq_label_ids': self.data[idx]['seq_label_ids'],
            'token_label_ids': self.data[idx]['token_label_ids'],
        }
        return sample


In [17]:
# 实例化自定义数据集
train_dataset = CAIDataset(train_data_lst)

# 使用DataLoader加载数据，batch_size为32
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)


In [18]:
optimizer = AdamW(model.parameters(), lr=args.lr, weight_decay=0.01)

/Users/bowie/anaconda3/envs/good/lib/python3.9/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [19]:
total_step = len(train_dataloader) * args.epoch
# y_total_step = len(val_dataloader) * args.epoch

train_losses = []
val_losses = []
global_step = 0

for epoch in range(args.epoch):
    model.train()
    train_loss = 0.0
    for batch in train_dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        seq_label_ids = batch['seq_label_ids'].to(device)
        token_label_ids = batch['token_label_ids'].to(device)

        optimizer.zero_grad()
        # print('text', batch['text'])
        intent_logits, slot_logits = model(input_ids, attention_mask, token_type_ids)

        # 定义意图分类的损失函数
        intent_loss_fn = torch.nn.CrossEntropyLoss()

        intent_loss = intent_loss_fn(intent_logits, seq_label_ids)


        # 定义槽位填充的损失函数
        slot_loss_fn = torch.nn.CrossEntropyLoss()

        slot_loss = slot_loss_fn(slot_logits.view(-1, len(args.slot_ner)), token_label_ids.view(-1))

        total_loss = intent_loss + slot_loss
        # print(total_loss, intent_loss, slot_loss)  # tensor(7.6379, grad_fn=<AddBackward0>) tensor(2.7916, grad_fn=<NllLossBackward0>) tensor(4.8463, grad_fn=<NllLossBackward0>)

        model.zero_grad()

        total_loss.backward()
        optimizer.step()

        print(f'[train] epoch:{epoch+1} {global_step}/{total_step} loss: {total_loss}')
        global_step += 1



[train] epoch:1 1/26350 loss: 0.04294595122337341
[train] epoch:1 2/26350 loss: 0.013517159037292004
[train] epoch:1 3/26350 loss: 0.023840121924877167
[train] epoch:1 4/26350 loss: 0.11999446898698807
[train] epoch:1 5/26350 loss: 0.10196840763092041
[train] epoch:1 6/26350 loss: 0.05742940679192543
[train] epoch:1 7/26350 loss: 1.8177517652511597
[train] epoch:1 8/26350 loss: 0.09658557176589966
[train] epoch:1 9/26350 loss: 0.07343640923500061
[train] epoch:1 10/26350 loss: 0.14284883439540863
[train] epoch:1 11/26350 loss: 2.2990617752075195
[train] epoch:1 12/26350 loss: 0.016192669048905373
[train] epoch:1 13/26350 loss: 0.05962982773780823
[train] epoch:1 14/26350 loss: 2.3599767684936523
[train] epoch:1 15/26350 loss: 0.07858696579933167
[train] epoch:1 16/26350 loss: 0.00700847152620554
[train] epoch:1 17/26350 loss: 0.296659916639328
[train] epoch:1 18/26350 loss: 0.027699723839759827
[train] epoch:1 19/26350 loss: 0.005785088986158371
[train] epoch:1 20/26350 loss: 0.0405368

In [20]:
# torch.save(model.state_dict(), 'pruned_model.pth')

In [21]:
# 永久去掉剪枝掩码，保存最终模型
# prune.remove(model.intent_classification[1], 'weight')
# prune.remove(model.slot_ner[1], 'weight')

Linear(in_features=768, out_features=121, bias=True)

In [22]:
# torch.save(model.state_dict(), 'pruned_model_no_mask.pth')

In [ ]:
# 加载剪枝后的模型示例
# model = MyModel()  # 重新定义模型
# model.load_state_dict(torch.load('pruned_model.pth'))
# # 去掉剪枝掩码
# prune.remove(model.fc1, 'weight')


In [24]:
with open('val.json', 'r', encoding="utf-8") as f:
    val_data = json.load(f)

val_data_lst = [tag_text(i) for i in val_data]

In [25]:
# 实例化自定义数据集
val_dataset = CAIDataset(val_data_lst)

# 使用DataLoader加载数据，batch_size为32
val_dataloader = DataLoader(val_dataset, batch_size=4, shuffle=True)


In [26]:
model.eval()
all_pred_intents = []
all_true_intents = []
all_pred_slots = []
all_true_slots = []
val_step = 1

with torch.no_grad():
    for batch in val_dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        seq_label_ids = batch['seq_label_ids'].to(device)  # 意图标签
        token_label_ids = batch['token_label_ids'].to(device)  # 槽位标签

        # 模型预测
        intent_logits, slot_logits = model(input_ids, attention_mask, token_type_ids)

        # 获取预测结果
        pred_intents = torch.argmax(intent_logits, dim=-1).cpu().numpy()  # 意图预测
        pred_slots = torch.argmax(slot_logits, dim=-1).cpu().numpy()  # 槽位预测

        all_pred_intents.extend(pred_intents)
        all_true_intents.extend(seq_label_ids.cpu().numpy())
        
        # 遍历每个样本并存储槽位预测
        for pred_slot, true_slot in zip(pred_slots, token_label_ids.cpu().numpy()):
            all_pred_slots.append(pred_slot[:len(true_slot)])  # 截取真实槽位长度部分
            all_true_slots.append(true_slot)

        val_step += 1
        print(val_step)
# 评估意图分类
intent_acc = accuracy_score(all_true_intents, all_pred_intents)
intent_prec, intent_rec, intent_f1, _ = precision_recall_fscore_support(all_true_intents, all_pred_intents, average='macro')

# 评估槽位填充
flat_pred_slots = [item for sublist in all_pred_slots for item in sublist]
flat_true_slots = [item for sublist in all_true_slots for item in sublist]
slot_prec, slot_rec, slot_f1, _ = precision_recall_fscore_support(flat_true_slots, flat_pred_slots, average='macro')

# 计算联合准确率
joint_correct = sum([p_int == t_int and np.array_equal(p_slot, t_slot) 
                        for p_int, p_slot, t_int, t_slot in zip(all_pred_intents, all_pred_slots, all_true_intents, all_true_slots)])
joint_accuracy = joint_correct / len(all_true_intents)

print({
    'intent_acc': intent_acc,
    'intent_prec': intent_prec,
    'intent_rec': intent_rec,
    'intent_f1': intent_f1,
    'slot_prec': slot_prec,
    'slot_rec': slot_rec,
    'slot_f1': slot_f1,
    'joint_acc': joint_accuracy
})

2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
{'intent_acc': 0.8866171003717472, 'intent_prec': 0.7266842464774187, 'intent_rec': 0.7617441742143081, 'intent_f1': 0.7214991420459778, 'slot_prec': 0.3431192618286725, 'slot_rec': 0.32060451323110484, 'slot_f1': 0.2972386520665045, 'joint_acc': 0.48141263940520446}


/Users/bowie/anaconda3/envs/good/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/bowie/anaconda3/envs/good/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/bowie/anaconda3/envs/good/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [ ]:
# # 下面是简单的例子 不知道能不能用 可以当伪代码参考
# import torch
# import torch.nn as nn
# import torch.nn.utils.prune as prune
# import torch.optim as optim
# import numpy as np

# # 定义简单的神经网络
# class SimpleNN(nn.Module):
#     def __init__(self):
#         super(SimpleNN, self).__init__()
#         self.fc1 = nn.Linear(10, 20)
#         self.fc2 = nn.Linear(20, 10)

#     def forward(self, x):
#         x = torch.relu(self.fc1(x))
#         x = self.fc2(x)
#         return x

# # 创建模型实例
# model = SimpleNN()

# # 随机生成一些训练数据
# X_train = torch.randn(100, 10)
# y_train = torch.randint(0, 10, (100,))

# # 定义损失函数和优化器
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.SGD(model.parameters(), lr=0.01)

# # 训练模型
# for epoch in range(5):  # 仅训练5个epoch以示范
#     model.train()
#     optimizer.zero_grad()
#     outputs = model(X_train)
#     loss = criterion(outputs, y_train)
#     loss.backward()
#     optimizer.step()
#     print(f'Epoch {epoch+1}, Loss: {loss.item()}')

# # 权重剪枝
# amount = 0.3  # 剪去30%的权重
# prune.l1_unstructured(model.fc1, name='weight', amount=amount)

# # 查看剪枝后的权重
# print("剪枝后的权重（fc1）：")
# print(model.fc1.weight)

# # 微调模型
# for epoch in range(5):  # 再微调5个epoch
#     model.train()
#     optimizer.zero_grad()
#     outputs = model(X_train)
#     loss = criterion(outputs, y_train)
#     loss.backward()
#     optimizer.step()
#     print(f'微调 Epoch {epoch+1}, Loss: {loss.item()}')

# # 保存剪枝后的模型
# torch.save(model.state_dict(), 'pruned_model.pth')

# # 永久去掉剪枝掩码
# prune.remove(model.fc1, 'weight')
